# Cut explorer — raw curves, per-$L$ locators, hand-driven FSS

One cut at a time, no registry, no automatic policy. Reuses `transition_fit.py`'s loaders
and locators (don't reimplement the math) but makes every decision explicit. Two parts,
because electric and magnetic cuts are genuinely different animals here:

- **Part 1 — electric cuts** (fixed $h_x$, sweep $h_z$): one curve per $L$, no branches,
  a single smooth-ish sigmoid. `O_FM_paratoric` is the order parameter everywhere on this
  line so far.
- **Part 2 — magnetic cuts** (fixed $h_z$, sweep $h_x$): every point belongs to an `up`
  chain (warm-started from the topological anchor) or a `dn` chain (from the polarized
  anchor); `load_runs` merges them into one lowest-energy winner curve, which is why a
  Part-1-style sigmoid fit on a magnetic cut came out jagged garbage earlier in this
  session. Use `load_runs_branched` and keep both branches visible instead. Compute BOTH
  the $O_\mathrm{FM}$-sigmoid locator (on the up branch only) and the energy-branch-crossing
  locator for every cut regardless of $h_z$ — don't assume up front which one "should" work;
  the data itself decides.

When a cut looks clean, copy its $h_c(\infty)$ into `phase_diagram_manual.ipynb` by hand.

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = (Path.cwd().parents[1] if Path.cwd().name == "notebooks" else Path.cwd()).resolve()
sys.path.insert(0, str(ROOT / "analysis" / "scripts"))
import transition_fit as tf

RES = ROOT / "results"
plt.rcParams.update({"figure.dpi": 120, "font.size": 11, "axes.spines.top": False,
                      "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.3})
print("ROOT =", ROOT)

# Part 1 — Electric cuts (fixed $h_x$, sweep $h_z$)

## 1 - Pick the cut

In [ ]:
HY = 0.0
FIXED = ("hx", 0.8)          # (field name, value) held fixed
SWEEP = "hz"                 # field being swept
OBS = "O_FM_paratoric"       # O_FM_paratoric (electric) | O_FM_membrane_R1 (magnetic)
LS = (4, 5, 6)
WINDOW = (0.1, 0.55)          # clip the curve to this range before fitting; None = no clip

CUT_DIR = "electric" if SWEEP == "hz" else "magnetic"
DIRS = [RES / "phase3d" / f"hy{HY}" / f"{CUT_DIR}_{FIXED[0]}{FIXED[1]}" / f"L{L}" for L in LS]
for d in DIRS:
    print(d, "exists" if d.exists() else "MISSING")

## 2 - Load and look at the raw observable (before any fitting)

In [ ]:
curves = tf.load_runs(DIRS, SWEEP, {FIXED[0]: FIXED[1], "hy": HY}, OBS)
COL = tf.plasma_by_L(list(curves))

fig, ax = plt.subplots(figsize=(5.5, 4))
for L, c in curves.items():
    ax.errorbar(c.h, c.y, 3 * np.nan_to_num(c.ye), fmt="o-", ms=5, lw=1, capsize=2,
                color=COL[L], label=f"L={L} (n={len(c.h)})")
if WINDOW and WINDOW[0] is not None:
    ax.axvspan(WINDOW[0], WINDOW[1], color="0.85", alpha=0.4, zorder=0, label="fit window")
ax.set_xlabel(f"$h_{{{SWEEP[1]}}}$"); ax.set_ylabel(OBS)
tf.openax(ax); ax.legend(frameon=False, fontsize=8)
fig.tight_layout(); plt.show()

## 3 - Per-$L$ locators, side by side

Nothing here is auto-combined into a single "central" value -- look at where logistic and
Richards agree or disagree before deciding what to trust. Richards refuses below 8 points
(6 free params, needs dof>=2; see session log) -- "no converge" at n=7 is that gate, not an
optimizer failure.

In [ ]:
FITS = {L: tf.locate_all(c, WINDOW) for L, c in curves.items()}

print(f"{'L':>2}  {'logistic':>18}  {'richards':>18}  {'fd_peak':>14}")
for L, fits in FITS.items():
    def fmt(f):
        return f"{f.h_c:.4f}+-{f.h_c_err:.4f} (chi2={f.chi2red:.1f})" if f.ok() else "no converge"
    print(f"{L:>2}  {fmt(fits['logistic']):>18}  {fmt(fits['richards']):>18}  {fmt(fits['fd_peak']):>14}")

fig, axes = plt.subplots(1, len(FITS), figsize=(4 * len(FITS), 3.5), sharey=True)
axes = np.atleast_1d(axes)
for ax, (L, c) in zip(axes, curves.items()):
    ax.errorbar(c.h, c.y, 3 * np.nan_to_num(c.ye), fmt="o", ms=5, color=COL[L], capsize=2, zorder=3)
    hh = np.linspace(c.h.min(), c.h.max(), 200)
    f = FITS[L]
    if f["logistic"].ok():
        ax.plot(hh, tf.logistic(hh, *f["logistic"].popt), "-", color="C0", lw=1.3, label="logistic")
        ax.axvline(f["logistic"].h_c, color="C0", ls=":", lw=1)
    if f["richards"].ok():
        ax.plot(hh, tf.richards(hh, *f["richards"].popt), ":", color="C1", lw=1.6, label="richards")
        ax.axvline(f["richards"].h_c, color="C1", ls=":", lw=1)
    ax.set_title(f"L={L}"); tf.openax(ax); ax.set_xlabel(f"$h_{{{SWEEP[1]}}}$")
axes[0].set_ylabel(OBS); axes[0].legend(frameon=False, fontsize=8)
fig.tight_layout(); plt.show()

## 4 - Pick $h_c(L)$ by hand

Default: use one method's value at every $L$ (`CENTRAL_METHOD`). Override individual sizes in
`MANUAL_HC` when the fit above is visibly untrustworthy -- put your own eyeballed `(value,
error)` there instead of trusting curve_fit blindly. **This dict does NOT auto-clear when you
change the cut above -- re-check/re-empty it every time you switch cuts, or you'll silently
refit a stale cut's numbers (happened once this session).**

In [ ]:
CENTRAL_METHOD = "logistic"      # "logistic" | "richards" | "fd_peak"
MANUAL_HC = {
    # 6: (0.30, 0.05),            # example override: L=6, h_c=0.30 +- 0.05
}

Ls, HC, HCE = [], [], []
for L, fits in FITS.items():
    if L in MANUAL_HC:
        hc, hce = MANUAL_HC[L]
    else:
        f = fits[CENTRAL_METHOD]
        if not f.ok():
            print(f"L={L}: {CENTRAL_METHOD} did not converge, skipping"); continue
        hc, hce = f.h_c, f.h_c_err
    Ls.append(L); HC.append(hc); HCE.append(hce)
    print(f"L={L}: h_c={hc:.4f} +- {hce:.4f}" + ("  [manual]" if L in MANUAL_HC else ""))
Ls, HC, HCE = map(np.array, (Ls, HC, HCE))

## 5 - FSS: play with the exponent

$h_c(L) = h_c(\infty) + a\,L^{-x}$. Common choices: `x=1.0` (bare 1/L / mean-field),
`x=1/nu_3DIsing=1.5874` (only justified where the transition is known/expected continuous
and 3D-Ising, i.e. established at $h_x=0$), `x=3.0` (volume law, closer to what a genuine
first-order pseudocritical shift should look like). **Set X below and re-run** -- don't
trust one exponent by default. x-axis is plain $1/L$ regardless of X; the fit line only
looks straight there when X=1 -- that curvature at other X is expected, not a bug.

In [ ]:
X = 1.5          # <- the knob. Try 1.0, 1.5874, 3.0, and anything between.
EXACT_REF = None     # e.g. 0.193869 at hx=0 -- set to draw a reference line, else None

fss = tf.fss_fit(Ls, HC, HCE, x=X)
print(f"x={X:.4f}  ->  h_c(inf) = {fss['h_inf']:.4f} +- {fss['h_inf_err']:.4f}   chi2red={fss['chi2red']:.3f}")

Lgrid = np.linspace(min(Ls), max(Ls) * 8, 200)
fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(1 / Lgrid, fss["h_inf"] + fss["a"] * Lgrid ** -X, "--", color="C0", lw=1.4, zorder=1)
for L, hc, hce in zip(Ls, HC, HCE):
    ax.errorbar([1 / L], [hc], [hce], fmt="o", ms=7, mfc=COL[L], mec="k", mew=0.4,
                ecolor="0.5", capsize=2, zorder=3, label=f"L={L}")
ax.errorbar([0], [fss["h_inf"]], [fss["h_inf_err"]], fmt="o", ms=9, mfc="C0", mec="k",
            mew=0.6, ecolor="C0", capsize=3, zorder=4, label="L=inf")
if EXACT_REF is not None:
    ax.axhline(EXACT_REF, color="k", ls=":", lw=1, label="exact")
ax.set_xlabel("$1/L$"); ax.set_ylabel("$h_c(L)$")
ax.legend(frameon=False, fontsize=8, loc="best")
fig.tight_layout(); plt.show()

### Sensitivity: how much does $h_c(\infty)$ move as $x$ varies?

In [ ]:
XS = np.linspace(0.5, 4.0, 36)
vals = [tf.fss_fit(Ls, HC, HCE, x=x) for x in XS]
hinf = np.array([v["h_inf"] for v in vals])
hinf_e = np.array([v["h_inf_err"] for v in vals])

fig, ax = plt.subplots(figsize=(5.5, 3.5))
ax.plot(XS, hinf, "-", color="C0", lw=1.3)
ax.fill_between(XS, hinf - hinf_e, hinf + hinf_e, color="C0", alpha=0.2, lw=0)
for x0, lbl in ((1.0, "x=1"), (1.5, "1/1.5"), (3.0, "x=3")):
    ax.axvline(x0, color="0.6", ls=":", lw=1)
    ax.text(x0, ax.get_ylim()[1], lbl, fontsize=7, ha="center", va="bottom")
if EXACT_REF is not None:
    ax.axhline(EXACT_REF, color="k", ls="--", lw=1, label="exact")
    ax.legend(frameon=False, fontsize=8)
ax.set_xlabel("FSS exponent x"); ax.set_ylabel(r"$h_c(\infty)$")
tf.openax(ax)
fig.tight_layout(); plt.show()
print("If this band swings past what you'd call the same answer as x varies, the cut isn't clean enough to quote yet.")

# Part 2 — Magnetic cuts (fixed $h_z$, sweep $h_x$)

Branch-aware from here on: `load_runs_branched` keeps the `up` and `dn` chains separate
instead of merging them, and two independent locators run on every cut:

- **$O_\mathrm{FM}$ sigmoid on the up branch** — only means what it looks like while the up
  branch is still tracking a real order-parameter melt and not a metastability spinodal.
- **Energy-branch crossing** (+ the $M_x$ = `sx_mean` jump evaluated at that crossing) — the
  first-order-transition locator: where the up and dn branches' energies cross is the
  equilibrium transition regardless of order, and the size of the $M_x$ jump there is a
  first-order-strength diagnostic.

Both run for every cut here, no matter the $h_z$ -- don't pre-assume "low $h_z$ = use
$O_\mathrm{FM}$, high $h_z$ = use the crossing" and only compute one. In the one cut checked
so far (`hz=0.0`, expected to be the cleanest topological-trivial case) they already
disagree by a lot: $O_\mathrm{FM}$(up) logistic gives L4=0.823, L5=0.888 (rising with L,
wrong direction for a converging pseudocritical point), while the energy crossing gives
L4=0.974, L5=0.975 (L-independent, and matching the exact anchor $h_x^c(h_z{=}0){=}1$ far
better). That's the up-branch spinodal, not the equilibrium point -- exactly what you'd
expect given `hz=0` is EXACTLY first order (CLAUDE.md's `hx_c=1.0` anchor). Worth deciding
per cut which number(s) you actually trust.

## 1 - Pick the cut

In [ ]:
HY = 0.0
HZ = 0.4                      # the fixed field
LS = (4, 5, 6)
WINDOW_OFM = (0.5, 1.3)        # window for the O_FM(up) sigmoid fit; None = no clip

DIRS_M = [RES / "phase3d" / f"hy{HY}" / f"magnetic_hz{HZ}" / f"L{L}" for L in LS]
for d in DIRS_M:
    print(d, "exists" if d.exists() else "MISSING")

## 2 - Load both branches, three observables, and look

In [ ]:
BR_OFM = tf.load_runs_branched(DIRS_M, "hx", {"hz": HZ, "hy": HY}, "O_FM_membrane_R1")
BR_E   = tf.load_runs_branched(DIRS_M, "hx", {"hz": HZ, "hy": HY}, "E0")
BR_MX  = tf.load_runs_branched(DIRS_M, "hx", {"hz": HZ, "hy": HY}, "sx_mean")
COL_M = tf.plasma_by_L(LS)

for br in ("up", "dn", "cold"):
    got = sorted(BR_OFM.get(br, {}))
    if got:
        print(f"branch={br}: L={got}")

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, (BR, ylab) in zip(axes, ((BR_OFM, "O_FM_membrane_R1"), (BR_E, "$E$"), (BR_MX, "$M_x$ (sx_mean)"))):
    for L in LS:
        for br, ls in (("up", "-"), ("dn", "--")):
            c = BR.get(br, {}).get(L)
            if c is None:
                continue
            y = c.E if BR is BR_E else c.y
            ax.plot(c.h, y, ls, marker="o", ms=4, color=COL_M[L], lw=1.3,
                    label=f"L={L} {br}" if ax is axes[0] else None)
    ax.set_xlabel("$h_x$"); ax.set_ylabel(ylab); tf.openax(ax)
axes[0].legend(frameon=False, fontsize=7, ncol=2)
fig.suptitle(f"hz={HZ}: up (solid) vs dn (dashed)", y=1.03)
fig.tight_layout(); plt.show()

## 3 - Both locators, per $L$

Missing branches at a given $L$ (e.g. an `up` chain that hasn't been launched/hasn't reached
far enough yet) just show as `--` -- that's real data-coverage information, not a bug.

In [ ]:
print(f"{'L':>2}  {'O_FM(up) logistic':>20}  {'E-crossing hx_c':>16}  {'Mx_up':>8}  {'Mx_dn':>8}  {'jump':>8}")
ROWS_M = []
for L in LS:
    up_ofm = BR_OFM.get("up", {}).get(L)
    ofm_str, ofm_hc, ofm_err = "--", np.nan, np.nan
    if up_ofm is not None:
        f = tf.locate_all(up_ofm, WINDOW_OFM)["logistic"]
        if f.ok():
            ofm_hc, ofm_err = f.h_c, f.h_c_err
            ofm_str = f"{f.h_c:.4f}+-{f.h_c_err:.4f}"

    up_e, dn_e = BR_E.get("up", {}).get(L), BR_E.get("dn", {}).get(L)
    hx_c, mx_up, mx_dn, jump = np.nan, np.nan, np.nan, np.nan
    ec_str = "--"
    if up_e is not None and dn_e is not None:
        hx_c, _ = tf.branch_crossing(up_e.h, up_e.E, dn_e.h, dn_e.E)
        if np.isfinite(hx_c):
            ec_str = f"{hx_c:.4f}"
            up_mx, dn_mx = BR_MX.get("up", {}).get(L), BR_MX.get("dn", {}).get(L)
            if up_mx is not None and dn_mx is not None:
                mx_up = float(np.interp(hx_c, up_mx.h, up_mx.y))
                mx_dn = float(np.interp(hx_c, dn_mx.h, dn_mx.y))
                jump = mx_dn - mx_up

    print(f"{L:>2}  {ofm_str:>20}  {ec_str:>16}  {mx_up:>8.4f}  {mx_dn:>8.4f}  {jump:>8.4f}")
    ROWS_M.append(dict(L=L, ofm_hc=ofm_hc, ofm_err=ofm_err, hx_c=hx_c, mx_jump=jump))

## 4 - FSS on whichever series you trust

Pick `SERIES` ("ofm" or "ecross") and an exponent `X`. Energy-crossing points are expected
to converge FAST with $L$ (first-order shift, not a critical exponent) -- if `X=1` already
gives a flat/consistent `h_inf` for the crossing series, that's doing its job; don't force
`1/nu_3DIsing` onto it just because Part 1 uses it there.

In [ ]:
SERIES = "ecross"     # "ofm" | "ecross"
X = 1.0

key = "hx_c" if SERIES == "ecross" else "ofm_hc"
errkey = None if SERIES == "ecross" else "ofm_err"
Ls_m = [r["L"] for r in ROWS_M if np.isfinite(r[key])]
hc_m = [r[key] for r in ROWS_M if np.isfinite(r[key])]
if SERIES == "ecross":
    # no per-L error model for the crossing yet -- use a flat placeholder (grid-scale) so
    # fss_fit has something to weight by; treat h_inf_err from this as a lower bound only.
    hce_m = [0.01] * len(hc_m)
else:
    hce_m = [r[errkey] for r in ROWS_M if np.isfinite(r[key])]

if len(Ls_m) >= 2:
    fss_m = tf.fss_fit(Ls_m, hc_m, hce_m, x=X)
    print(f"series={SERIES} x={X}  ->  h_inf={fss_m['h_inf']:.4f}+-{fss_m['h_inf_err']:.4f}  (n={len(Ls_m)} sizes: {Ls_m})")
else:
    print(f"series={SERIES}: only {len(Ls_m)} usable size(s) ({Ls_m}) -- not enough to extrapolate yet")